# MolForge GRPO Training Pipeline
This notebook implements the Reinforcement Learning (GRPO) training pipeline for the MolForge environment.
We train the model using a **Proposer-Critic-Selector** architecture and targeted **reward shaping** to overcome local minima.

In [ ]:
!pip install -U "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -U "trl>=0.21.0" peft accelerate bitsandbytes datasets matplotlib pandas huggingface_hub "openenv-core[core]>=0.2.3" rdkit jmespath xformers

In [ ]:
import os
import sys
from pathlib import Path

# Clone the repository
if not Path("/content/molt_lab").exists():
    !git clone https://github.com/Adhitya-Vardhan/molt_lab.git /content/molt_lab

# Add project root to path
if "/content/molt_lab" not in sys.path:
    sys.path.insert(0, "/content/molt_lab")
    
# Change working directory
os.chdir("/content/molt_lab")

In [ ]:
import time
import os

# Training Configuration
os.environ["MOLFORGE_REWARD_MODE"] = "curriculum"
os.environ["MOLFORGE_TRAINING_RANDOMIZATION"] = "1"

RL_MAX_STEPS = 300
NUM_GENERATIONS = 2
PER_DEVICE_BATCH = 2
GRAD_ACCUM = 4
LEARNING_RATE = 2e-6
MAX_SEQ_LENGTH = 2048
MAX_PROMPT_LENGTH = 1536
MAX_COMPLETION_LENGTH = 384

RUN_NAME = time.strftime("molforge_grpo_%Y%m%d_%H%M%S")
OUTPUT_DIR = Path(f"/content/molforge_rl_runs/{RUN_NAME}")
ADAPTER_SAVE_DIR = OUTPUT_DIR / "adapters"
PLOT_DIR = OUTPUT_DIR / "plots"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR = OUTPUT_DIR / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)


### Reward Function & OpenEnv Integration
We implement a custom reward function that wraps the native `MolForgeEnvironment`. 
To prevent "reward hacking" (where the model endlessly farms `run_assay` for safe points), we apply targeted reward shaping.

In [ ]:
import json
import time
from typing import Any, Dict, Tuple
from inference_common import MolForgeAction, attach_reasoning_fields, attach_team_messages, extract_json
from server.molforge_environment import MolForgeEnvironment

COMPLETION_LOG = LOG_DIR / "completion_diagnostics.jsonl"

def replay_to_state(record: dict[str, Any]) -> MolForgeEnvironment:
    env = MolForgeEnvironment()
    if record.get("randomized"): os.environ["MOLFORGE_TRAINING_RANDOMIZATION"] = "1"
    os.environ["MOLFORGE_RAND_SEED"] = str(record.get("random_seed", "rl"))
    observation = env.reset()
    for action_payload in record.get("pre_actions", []):
        action = MolForgeAction(**action_payload)
        observation = env.step(attach_team_messages(observation, attach_reasoning_fields(observation, action)))
    return env

def evaluate_completion(prompt_str, completion_str, record) -> Tuple[float, dict]:
    try:
        action_dict = extract_json(completion_str)
        action = MolForgeAction(**action_dict)
        valid_json = True
    except Exception:
        return -1.5, {"valid_json": False, "action_type": "invalid"}

    env = replay_to_state(record)
    observation = env._build_observation(reward=0.0, done=False, reward_components=[])
    action = attach_team_messages(observation, attach_reasoning_fields(observation, action))
    next_observation = env.step(action)
    
    # --- ANTI-REWARD HACKING FILTER ---
    # We manually sum only the scientific reward components, ignoring "chatter" rewards
    filtered_reward = 0.0
    keep_components = {
        "edit_delta", "submission_quality", "hard_constraints", "baseline_gate",
        "submission_evidence", "curriculum_terminal_progress", "curriculum_evidence_gate"
    }
    penalties = {"invalid_action", "budget_exhausted", "step_limit", "policy_veto", "loop_penalty"}
    
    for component in next_observation.reward_components:
        if component.name in keep_components:
            filtered_reward += component.value
        elif component.name in penalties:
            filtered_reward += component.value

    # Add a mandatory time pressure penalty for every step
    filtered_reward -= 0.15 
    
    grader_scores = next_observation.metadata.get("terminal_grader_scores", {})
    
    # Extra multipliers for reaching the goal
    if action.action_type == "submit" and grader_scores.get("submission_score", 0) > 0:
        filtered_reward += float(grader_scores["submission_score"]) * 4.0
    
    reward = round(filtered_reward, 4)
    
    return reward, {
        "valid_json": True, "action_type": action.action_type, "reward": reward, 
        "done": next_observation.done, "scores": grader_scores, 
        "raw_completion": completion_str, "timestamp": time.time()
    }

def molforge_reward_func(prompts, completions, **kwargs) -> list[float]:
    rewards = []
    for i in range(len(completions)):
        record = {"pre_actions": kwargs["record"][i]["pre_actions"] if "record" in kwargs else []}
        reward, diagnostics = evaluate_completion("", completions[i][0]["content"], record)
        rewards.append(reward)
        with open(COMPLETION_LOG, "a") as f:
            f.write(json.dumps(diagnostics) + "\n")
    return rewards


### Model & Tokenizer Loading

In [ ]:
from unsloth import FastLanguageModel

# Set this to your SFT checkpoint Deployed to hugging face space 
# SFT trained on only to mimic the response behavioiur of the model (structured responses visit the hf blog for more detailed explanation )
SFT_ADAPTER_PATH = "Adhitya122/qwen3_5_2b_molforge_sft_v4"

print("Loading model and applying Unsloth optimizations...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_ADAPTER_PATH,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

# Enable fast training paths
FastLanguageModel.for_training(model)

# Extract underlying tokenizer if it is wrapped in a vision processor
if hasattr(tokenizer, "tokenizer"):
    tokenizer = tokenizer.tokenizer

### GRPO Training Loop

In [ ]:
from datasets import Dataset
from scripts.generate_sft_compact_policy_v4_dataset import compact_action_payload, COMPACT_ACTION_SYSTEM_PROMPT
from inference_common import heuristic_team_action
import random

def build_dynamic_prompts(episodes=50, max_turns=5) -> Dataset:
    """Generates training prompts by playing the environment with a heuristic expert."""
    print(f"Generating {episodes} episodes of dynamic prompts...")
    records = []
    env = MolForgeEnvironment()
    
    for _ in range(episodes):
        observation = env.reset()
        pre_actions = []
        
        for _ in range(max_turns):
            if observation.done:
                break
            
            # Capture the current state as a prompt
            prompt_payload = compact_action_payload(observation)
            records.append({
                "prompt": [
                    {"role": "system", "content": COMPACT_ACTION_SYSTEM_PROMPT},
                    {"role": "user", "content": json.dumps(prompt_payload)}
                ],
                "record": {
                    "scenario_id": observation.scenario_id,
                    "difficulty": observation.difficulty,
                    "step_index": observation.step_index,
                    "pre_actions": list(pre_actions),
                    "randomized": True,
                    "random_seed": "dynamic-rl"
                }
            })
            
            # Use expert to move to the next state
            action = heuristic_team_action(observation)
            observation = env.step(action)
            pre_actions.append({"action_type": action.action_type, "acting_role": action.acting_role})
            
    random.shuffle(records)
    return Dataset.from_list(records)

# Generate the dataset dynamically (no .jsonl needed!)
dataset = build_dynamic_prompts(episodes=20, max_turns=6)
print(f"Dynamic dataset created with {len(dataset)} prompt states.")


In [ ]:
from trl import GRPOConfig, GRPOTrainer
import inspect
import torch

# Check for BF16 support (T4 does not support it, A100/L4 do)
has_bf16 = torch.cuda.is_bf16_supported()
print(f"GPU supports BF16: {has_bf16}")

config_kwargs = {
    "output_dir": str(OUTPUT_DIR),
    "learning_rate": LEARNING_RATE,
    "per_device_train_batch_size": PER_DEVICE_BATCH,
    "gradient_accumulation_steps": GRAD_ACCUM,
    "max_prompt_length": MAX_PROMPT_LENGTH,
    "max_completion_length": MAX_COMPLETION_LENGTH,
    "num_generations": NUM_GENERATIONS,
    "max_steps": RL_MAX_STEPS,
    "logging_steps": 1,
    "save_steps": 25,
    "bf16": has_bf16,
    "fp16": not has_bf16,
    "report_to": "none",
    "log_completions": True,
}

supported_params = inspect.signature(GRPOConfig.__init__).parameters
filtered_kwargs = {k: v for k, v in config_kwargs.items() if k in supported_params}

training_args = GRPOConfig(**filtered_kwargs)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=molforge_reward_func,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print("Starting GRPO Training...")
trainer.train()

print(f"Training complete. Saving adapters to {ADAPTER_SAVE_DIR}")
trainer.save_model(str(ADAPTER_SAVE_DIR))
tokenizer.save_pretrained(str(ADAPTER_SAVE_DIR))
